In [113]:
import pandas as pd
import os

In [114]:
# Load data
df = pd.read_csv('../data/raw/Unemployment_data.csv')

print(df.head())
df.shape

           Region         Date  Frequency   Estimated Unemployment Rate (%)  \
0  Andhra Pradesh   31-05-2019    Monthly                              3.65   
1  Andhra Pradesh   30-06-2019    Monthly                              3.05   
2  Andhra Pradesh   31-07-2019    Monthly                              3.75   
3  Andhra Pradesh   31-08-2019    Monthly                              3.32   
4  Andhra Pradesh   30-09-2019    Monthly                              5.17   

    Estimated Employed   Estimated Labour Participation Rate (%)   Area  
0           11999139.0                                     43.24  Rural  
1           11755881.0                                     42.05  Rural  
2           12086707.0                                     43.50  Rural  
3           12285693.0                                     43.97  Rural  
4           12256762.0                                     44.68  Rural  


(768, 7)

In [115]:
# Remove whitspaces in the columns
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(' ', '_')
print(df.columns.tolist())


['Region', 'Date', 'Frequency', 'Estimated_Unemployment_Rate_(%)', 'Estimated_Employed', 'Estimated_Labour_Participation_Rate_(%)', 'Area']


In [116]:
# Dropping Nulls
df.dropna(inplace=True)

print(df.isnull().sum())
print(f'\n{'='*50}')
print("Duplicate rows:", df.duplicated().sum())
print(f'{'='*50}')
print(f'Shape after dropping: {df.shape}')



Region                                     0
Date                                       0
Frequency                                  0
Estimated_Unemployment_Rate_(%)            0
Estimated_Employed                         0
Estimated_Labour_Participation_Rate_(%)    0
Area                                       0
dtype: int64

Duplicate rows: 0
Shape after dropping: (740, 7)


In [117]:
print(df['Frequency'].value_counts())

df['Frequency'] = df['Frequency'].str.strip()

print(f'\n{'='*50}')
print(df['Frequency'].value_counts())

Frequency
Monthly     381
 Monthly    359
Name: count, dtype: int64

Frequency
Monthly    740
Name: count, dtype: int64


In [118]:
df.duplicated(subset=['Region', 'Date', 'Area']).any()

np.False_

In [119]:
# Converting Date to datetime typw
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

print(df['Date'].dtype)


datetime64[us]


In [120]:
region_area_counts = df.groupby(['Region', 'Area']).size().unstack(fill_value=0)
region_area_counts['Total'] = region_area_counts.sum(axis=1)
region_area_counts['Missing (of 28 expected)'] = 28 - region_area_counts['Total']

print(region_area_counts.sort_values('Missing (of 28 expected)', ascending=False))

Area              Rural  Urban  Total  Missing (of 28 expected)
Region                                                         
Chandigarh            0     12     12                        16
Sikkim                5     12     17                        11
Jammu & Kashmir      11     10     21                         7
Goa                  12     12     24                         4
Puducherry           12     14     26                         2
Assam                12     14     26                         2
Meghalaya            14     13     27                         1
Uttarakhand          13     14     27                         1
Bihar                14     14     28                         0
Andhra Pradesh       14     14     28                         0
Delhi                14     14     28                         0
Chhattisgarh         14     14     28                         0
Jharkhand            14     14     28                         0
Himachal Pradesh     14     14     28   

### Incomplete Region Reporting

20 of 28 regions have complete reporting (14 Rural + 14 Urban = 28 rows). 8 regions
have gaps:

| Region | Rural | Urban | Missing | Detail |
|--------|-------|-------|---------|--------|
| Chandigarh | 0/14 | 12/14 | 16 | Rural entirely absent; Urban also short by 2 |
| Sikkim | 5/14 | 12/14 | 11 | Rural mostly missing (9 gaps); Urban short by 2 |
| Jammu & Kashmir | 11/14 | 10/14 | 7 | Both areas short, roughly evenly |
| Goa | 12/14 | 12/14 | 4 | Both areas short by 2 |
| Puducherry | 12/14 | 14/14 | 2 | Rural short by 2; Urban complete |
| Assam | 12/14 | 14/14 | 2 | Rural short by 2; Urban complete |
| Meghalaya | 14/14 | 13/14 | 1 | Rural complete; Urban short by 1 |
| Uttarakhand | 13/14 | 14/14 | 1 | Rural short by 1; Urban complete |

**Decision:** gaps are preserved as-is rather than imputed. Fabricating values
would compromise the integrity of the COVID-period analysis this project is
built around. Any region-level comparison or "national average" in later
notebooks should account for these gaps — Chandigarh and Sikkim in particular
have Rural and Urban row counts different enough that a naive combined average
would be skewed by whichever area is over-represented for that region.

In [121]:
print(df['Region'].unique())
print(f'\n{'='*50}')
print(df['Area'].unique())

<ArrowStringArray>
[  'Andhra Pradesh',            'Assam',            'Bihar',
     'Chhattisgarh',            'Delhi',              'Goa',
          'Gujarat',          'Haryana', 'Himachal Pradesh',
  'Jammu & Kashmir',        'Jharkhand',        'Karnataka',
           'Kerala',   'Madhya Pradesh',      'Maharashtra',
        'Meghalaya',           'Odisha',       'Puducherry',
           'Punjab',        'Rajasthan',           'Sikkim',
       'Tamil Nadu',        'Telangana',          'Tripura',
    'Uttar Pradesh',      'Uttarakhand',      'West Bengal',
       'Chandigarh']
Length: 28, dtype: str

<ArrowStringArray>
['Rural', 'Urban']
Length: 2, dtype: str


In [122]:
df.drop(columns='Frequency', inplace=True)
df.shape

(740, 6)

In [123]:
print('--- Final clean overview ----\n')
print(f'Rows:    {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'\n{'='*50}')
print(f'Date range: {df['Date'].min()} to {df['Date'].max()}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'{'='*50}\n')
print(f'Columns dtypes:\n{df.dtypes}')



--- Final clean overview ----

Rows:    740
Columns: 6

Date range: 2019-05-31 00:00:00 to 2020-06-30 00:00:00
Missing values: 0

Columns dtypes:
Region                                                str
Date                                       datetime64[us]
Estimated_Unemployment_Rate_(%)                   float64
Estimated_Employed                                float64
Estimated_Labour_Participation_Rate_(%)           float64
Area                                                  str
dtype: object


In [124]:
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/unemployment_clean.csv', index=False)
print('Saved to ../data/processed/unemployment_clean.csv')
print(f'File size: {os.path.getsize('../data/processed/unemployment_clean.csv')}')

Saved to ../data/processed/unemployment_clean.csv
File size: 36670


## Cleaning Summary

### Missing values
28 fully empty rows dropped (768 → 740 rows), matching the count found in
01_data_understanding.ipynb.

### Duplicates
0 genuine duplicate rows after dropping empty rows. The 27 rows initially
flagged by `.duplicated()` were entirely the empty rows themselves.

### Frequency column
Values had inconsistent leading whitespace ('Monthly' vs ' Monthly', 381/359
split) masking that the column is constant across all 740 rows. Cleaned with
`.str.strip()`, confirmed zero variance, then dropped.

### Date
Converted from string ('31-05-2019') to proper datetime64 using dayfirst=True.

### Incomplete region reporting
20 of 28 regions have complete 28-row coverage (14 months x Rural + Urban).
8 regions have gaps, most notably Chandigarh (Rural entirely absent) and
Sikkim (Rural mostly absent). Verified via a systematic groupby rather than
manual inspection, which caught two count errors in the initial by-eye
investigation. Gaps preserved as-is — not imputed — to protect the integrity
of the COVID-period analysis.

### Final dataset
740 rows x 6 columns (down from the original 7 — Frequency was dropped after
being confirmed constant across every row, see above). 0 missing values.
Saved to ../data/processed/unemployment_clean.csv.

### Next step
Exploratory data analysis — unemployment trends over time, Rural vs Urban
comparison, and regional distribution, accounting for the incomplete regions
noted above.